# Transformer Embedding Structure Experiments

Compares **E5** vs **MiniLM** and **separate** vs **combined** pair encoding,
with XGBoost + Optuna (5 trials per experiment).

1. **E5** (`intfloat/multilingual-e5-small`, aligned with notebook 06) + combined pair encoding, `passage:` prefix
2. **MiniLM** + separate premise/hypothesis + `|p-h|`, `p*h` (mean pooling)
3. **MiniLM** + combined pair encoding (mean pooling)



## Setup (run once before experiments)

In [ ]:
from pathlib import Path

import joblib
import mlflow
import numpy as np
import pandas as pd
import optuna
import torch
from transformers import AutoModel, AutoTokenizer
from xgboost import XGBClassifier

from helpers.data import load_processed_data, load_sample_submission
from helpers.metrics import compute_all_metrics
from helpers.mlflow import log_common_context, log_metrics, log_model_artifacts, start_notebook_run
from helpers.submission import build_submission, save_submission

optuna.logging.set_verbosity(optuna.logging.WARNING)

PROCESSED_DIR = Path('../data/processed')
RAW_DIR = Path('../data/raw')

train_df, val_df, test_df = load_processed_data(PROCESSED_DIR)
sample_submission = load_sample_submission(RAW_DIR / 'sample_submission.csv')

y_train = train_df['label'].astype(int)
y_val = val_df['label'].astype(int)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

OLD_MODEL_NAME = 'intfloat/multilingual-e5-small'
NEW_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
BATCH_SIZE = 64
MAX_LENGTH = 128
OPTUNA_TRIALS = 5

results_table = []


def free_gpu(*models) -> None:
    for model in models:
        del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def apply_e5_prefix(texts: list[str], model_name: str, prefix: str = 'passage') -> list[str]:
    if 'e5' in model_name.lower():
        return [f'{prefix}: {text}' for text in texts]
    return list(texts)


def mean_pooling(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


def pool_embeddings(outputs, inputs: dict, model_name: str) -> torch.Tensor:
    if 'e5' in model_name.lower():
        return outputs.last_hidden_state[:, 0, :]
    return mean_pooling(outputs.last_hidden_state, inputs['attention_mask'])


def get_separate_embeddings(texts, model, tokenizer, model_name: str, batch_size: int = BATCH_SIZE):
    model.eval()
    prefixed = apply_e5_prefix(texts, model_name, prefix='passage')
    all_embs = []
    for start in range(0, len(prefixed), batch_size):
        batch = prefixed[start : start + batch_size]
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors='pt',
        ).to(device)
        with torch.no_grad():
            outputs = model(**inputs)
            embs = pool_embeddings(outputs, inputs, model_name).cpu().numpy()
            all_embs.append(embs)
    return np.vstack(all_embs)


def prepare_features_separate(premise_embs, hypothesis_embs):
    return np.hstack([premise_embs, hypothesis_embs, np.abs(premise_embs - hypothesis_embs), premise_embs * hypothesis_embs])


def get_combined_embeddings(premises, hypotheses, model, tokenizer, model_name: str, batch_size: int = BATCH_SIZE):
    model.eval()
    batch_p = apply_e5_prefix(premises, model_name, prefix='passage')
    batch_h = apply_e5_prefix(hypotheses, model_name, prefix='passage')
    all_embeddings = []
    for start in range(0, len(batch_p), batch_size):
        p_slice = batch_p[start : start + batch_size]
        h_slice = batch_h[start : start + batch_size]
        inputs = tokenizer(
            p_slice,
            h_slice,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors='pt',
        ).to(device)
        with torch.no_grad():
            outputs = model(**inputs)
            embs = pool_embeddings(outputs, inputs, model_name).cpu().numpy()
            all_embeddings.append(embs)
    return np.vstack(all_embeddings)


def run_xgboost_experiment(x_train, x_val, x_test, run_name: str, feature_tag: str):
    def objective_xgb(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 250),
            'max_depth': trial.suggest_int('max_depth', 3, 8),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'random_state': 42,
            'eval_metric': 'mlogloss',
            'n_jobs': -1,
            'tree_method': 'hist',
        }
        model = XGBClassifier(**params)
        model.fit(x_train, y_train)
        return model.score(x_val, y_val)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective_xgb, n_trials=OPTUNA_TRIALS, show_progress_bar=True)

    best_model = XGBClassifier(
        **study.best_params,
        random_state=42,
        eval_metric='mlogloss',
        n_jobs=-1,
        tree_method='hist',
    )
    best_model.fit(x_train, y_train)

    val_preds = best_model.predict(x_val)
    metrics = compute_all_metrics(val_df, val_preds)
    test_preds = best_model.predict(x_test)

    submission_path = Path('../submissions') / f'{run_name}.csv'
    save_submission(build_submission(sample_submission, test_preds), submission_path)
    model_path = Path('../submissions') / f'{run_name}_model.joblib'
    joblib.dump(best_model, model_path)

    with start_notebook_run(
        run_name,
        tags={'model_family': 'xgboost', 'features': feature_tag, 'optimized': 'optuna'},
    ):
        mlflow.log_params(study.best_params)
        log_metrics(metrics)
        log_common_context('../configs/data_split.yaml', '../data/processed/split_metadata.json')
        mlflow.log_artifact(str(submission_path), artifact_path='submissions')
        log_model_artifacts(artifacts={'model.joblib': model_path}, artifact_path='model')

    print(
        f'Результати {run_name}: accuracy={metrics["accuracy"]:.4f}, '
        f'f1_macro={metrics["f1_macro"]:.4f}'
    )
    return {
        'experiment': run_name,
        'feature_tag': feature_tag,
        'accuracy': metrics['accuracy'],
        'f1_macro': metrics['f1_macro'],
        'best_params': study.best_params,
    }

## Experiment 1: E5 + combined structure

In [ ]:
print('--- Experiment 1: E5 + combined ---')
tokenizer_old = AutoTokenizer.from_pretrained(OLD_MODEL_NAME)
model_old = AutoModel.from_pretrained(OLD_MODEL_NAME).to(device)

print('Generating combined embeddings for E5...')
x_train_exp1 = get_combined_embeddings(
    train_df['premise'].tolist(),
    train_df['hypothesis'].tolist(),
    model_old,
    tokenizer_old,
    OLD_MODEL_NAME,
)
x_val_exp1 = get_combined_embeddings(
    val_df['premise'].tolist(),
    val_df['hypothesis'].tolist(),
    model_old,
    tokenizer_old,
    OLD_MODEL_NAME,
)
x_test_exp1 = get_combined_embeddings(
    test_df['premise'].tolist(),
    test_df['hypothesis'].tolist(),
    model_old,
    tokenizer_old,
    OLD_MODEL_NAME,
)

res1 = run_xgboost_experiment(
    x_train_exp1,
    x_val_exp1,
    x_test_exp1,
    'e5_new_combined_structure',
    'e5_combined',
)
results_table.append(res1)

free_gpu(model_old)
del tokenizer_old

## Experiment 2: MiniLM + separate structure

In [ ]:
print('--- Experiment 2: MiniLM + separate ---')
tokenizer_new = AutoTokenizer.from_pretrained(NEW_MODEL_NAME)
model_new = AutoModel.from_pretrained(NEW_MODEL_NAME).to(device)

print('Generating separate embeddings for MiniLM...')
train_p = get_separate_embeddings(train_df['premise'].tolist(), model_new, tokenizer_new, NEW_MODEL_NAME)
train_h = get_separate_embeddings(train_df['hypothesis'].tolist(), model_new, tokenizer_new, NEW_MODEL_NAME)
val_p = get_separate_embeddings(val_df['premise'].tolist(), model_new, tokenizer_new, NEW_MODEL_NAME)
val_h = get_separate_embeddings(val_df['hypothesis'].tolist(), model_new, tokenizer_new, NEW_MODEL_NAME)
test_p = get_separate_embeddings(test_df['premise'].tolist(), model_new, tokenizer_new, NEW_MODEL_NAME)
test_h = get_separate_embeddings(test_df['hypothesis'].tolist(), model_new, tokenizer_new, NEW_MODEL_NAME)

x_train_exp2 = prepare_features_separate(train_p, train_h)
x_val_exp2 = prepare_features_separate(val_p, val_h)
x_test_exp2 = prepare_features_separate(test_p, test_h)

res2 = run_xgboost_experiment(
    x_train_exp2,
    x_val_exp2,
    x_test_exp2,
    'miniLM_old_separate_structure',
    'miniLM_separate',
)
results_table.append(res2)

## Experiment 3: MiniLM + combined structure

In [ ]:
print('--- Experiment 3: MiniLM + combined ---')

print('Generating combined embeddings for MiniLM...')
x_train_exp3 = get_combined_embeddings(
    train_df['premise'].tolist(),
    train_df['hypothesis'].tolist(),
    model_new,
    tokenizer_new,
    NEW_MODEL_NAME,
)
x_val_exp3 = get_combined_embeddings(
    val_df['premise'].tolist(),
    val_df['hypothesis'].tolist(),
    model_new,
    tokenizer_new,
    NEW_MODEL_NAME,
)
x_test_exp3 = get_combined_embeddings(
    test_df['premise'].tolist(),
    test_df['hypothesis'].tolist(),
    model_new,
    tokenizer_new,
    NEW_MODEL_NAME,
)

res3 = run_xgboost_experiment(
    x_train_exp3,
    x_val_exp3,
    x_test_exp3,
    'miniLM_new_combined_structure',
    'miniLM_combined',
)
results_table.append(res3)

free_gpu(model_new)
del tokenizer_new

## Final comparison

In [ ]:
summary = pd.DataFrame(results_table).sort_values('f1_macro', ascending=False)
results_path = Path('../submissions') / 'transformer_structure_optuna_comparison.csv'
summary.to_csv(results_path, index=False)
print(f'Saved comparison table to {results_path}')
summary